# 🌍 Week 5 — Spatial Libraries in Python
### Geospatial Python Mastery | Module C: Spatial Python Stack

---

|  |  |
|---|---|
| **Course** | Geospatial Python Mastery |
| **Week** | 5 of 10 |
| **Theme** | Spatial Libraries in Python |
| **Duration** | ~4 contact hours + 4 hours self-study |
| **Practice Outcome** | Lab: analyse points, lines, and polygons for a local study area |
| **Next week** | PostgreSQL connection and data loading |

---

> 🗺️ **Why this week matters**
> Weeks 1–4 gave you Python fundamentals, data structures, file processing, and CRS theory.
> Week 5 is where everything clicks into place: **GeoPandas** lets you treat geospatial
> data like a spreadsheet; **Shapely** gives you a geometry toolkit; **PyProj** handles
> coordinate transforms; **Fiona** reads and writes practically every vector format; and
> **Rasterio** opens the door to gridded / raster data.
> By the end of this week, you will run a complete spatial analysis pipeline — loading data,
> transforming coordinates, measuring distances and areas, joining datasets, and producing
> both static and interactive maps.

---

## 📋 Table of Contents

| Section | Topic |
|---------|-------|
| **1** | GeoPandas — GeoDataFrame Basics |
| **2** | Shapely — Geometry Types and Operations |
| **3** | PyProj — CRS and Coordinate Transformations |
| **4** | Spatial Measurements — Distance, Area, Length |
| **5** | Spatial Joins and Overlays |
| **6** | Fiona — Reading and Writing Vector Files |
| **7** | Rasterio — Raster Data Introduction |
| **8** | Visualization — Matplotlib and Folium Maps |
| **🔬** | Mini-Lab: Local Study Area Analysis |
| **✅** | Summary, Checklist, Week 6 Preview |

---

## 🗺️ Symbol Guide

| Symbol | Meaning |
|--------|---------|
| 📖 | Concept explanation — read carefully |
| 💻 | Live code cell — run it and study the output |
| 🎯 | Exercise — complete the code yourself |
| 💡 | Tip — best-practice advice |
| ⚠️ | Warning — common mistake |
| 🔬 | Mini-Lab — longer hands-on project |
| ✅ | Solution cell (uncomment to reveal) |

> **Keyboard shortcuts:** `Shift+Enter` run & move · `Ctrl+Enter` run in place ·
> `b` insert cell below · `m` convert to Markdown · `Esc` command mode

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

| # | Objective |
|---|-----------|
| 1 | Create and manipulate **GeoDataFrames** with GeoPandas |
| 2 | Perform geometry operations (buffer, union, intersection) using **Shapely** |
| 3 | Reproject data between coordinate systems with **PyProj** |
| 4 | Measure distances, areas, and lengths accurately in a projected CRS |
| 5 | Run **spatial joins** (point-in-polygon, nearest) and **overlays** |
| 6 | Read and write shapefiles, GeoJSON, and GeoPackages with **Fiona** |
| 7 | Open a raster file with **Rasterio** and extract band statistics |
| 8 | Produce a static map with **Matplotlib** and an interactive map with **Folium** |

> 💡 Every example uses a consistent study area — a set of European city points, road
> corridors, and administrative polygons — so you can trace data through all eight sections.

### How to use this notebook
* Run cells **in order** — later cells depend on names defined earlier.
* 🎯 cells are **exercises** — attempt them before looking at the ✅ solution.
* Comments (`#`) explain individual lines; prose explanation appears in markdown cells above.

In [ ]:
# 💻 Environment check and auto-install
# ─────────────────────────────────────────────────────────────────────────────
import sys
import subprocess
import importlib
import platform

try:
    import google.colab          # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

OS_NAME = platform.system()

print("=" * 60)
print("  Geospatial Python Mastery — Week 5 Environment Check")
print("=" * 60)
print(f"  Environment : {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"  OS          : {OS_NAME} {platform.release()}")
print(f"  Python      : {sys.version.split()[0]}")
print("=" * 60)

REQUIRED = [
    ("geopandas",  "geopandas",  "0.14"),
    ("shapely",    "shapely",    "2.0"),
    ("pyproj",     "pyproj",     "3.6"),
    ("fiona",      "fiona",      "1.9"),
    ("rasterio",   "rasterio",   "1.3"),
    ("numpy",      "numpy",      "1.25"),
    ("matplotlib", "matplotlib", "3.8"),
    ("folium",     "folium",     "0.15"),
]

print("\nChecking packages:")
for import_name, pip_name, min_ver in REQUIRED:
    try:
        mod = importlib.import_module(import_name)
        ver = getattr(mod, "__version__", "?")
        print(f"  ✅  {pip_name:<15} v{ver}")
    except ImportError:
        print(f"  📦  Installing {pip_name}...", end=" ", flush=True)
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", pip_name],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print("done ✅")
        else:
            print("FAILED ❌")
            print(f"      stderr: {result.stderr[-300:]}")

if OS_NAME == "Windows" and not IN_COLAB:
    print("\n💡 Windows tip: if fiona/rasterio failed, use:")
    print("   conda install -c conda-forge geopandas rasterio fiona")

print("\n✅ Environment check complete — ready for Week 5!")

In [ ]:
# 💻 Import all libraries used this week
# ─────────────────────────────────────────────────────────────────────────────
import math
import json
import logging
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

import geopandas as gpd
from shapely.geometry import (
    Point, MultiPoint,
    LineString, MultiLineString,
    Polygon, MultiPolygon,
    mapping, shape
)
from shapely import wkt, affinity
from shapely.ops import unary_union, split
import pyproj
from pyproj import CRS, Transformer, Geod
import fiona
import rasterio
from rasterio.transform import from_bounds
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium

warnings.filterwarnings("ignore", category=FutureWarning)  # keep output clean

print(f"geopandas  : {gpd.__version__}")
print(f"shapely    : {__import__('shapely').__version__}")
print(f"pyproj     : {pyproj.__version__}")
print(f"fiona      : {fiona.__version__}")
print(f"rasterio   : {rasterio.__version__}")
print(f"numpy      : {np.__version__}")
print(f"matplotlib : {matplotlib.__version__}")
print(f"folium     : {folium.__version__}")
print("\n🎉 All packages imported successfully!")

In [ ]:
# 💻 Create local data directories (idempotent)
DATA_DIR = Path("data/week_05")
RAW_DIR  = DATA_DIR / "raw"
OUT_DIR  = DATA_DIR / "output"
LOGS_DIR = DATA_DIR / "logs"

for p in [RAW_DIR, OUT_DIR, LOGS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Directories ready:")
for p in [RAW_DIR, OUT_DIR, LOGS_DIR]:
    print(f"  {p}")

---
## 📖 Section 1 — GeoPandas: GeoDataFrame Basics

GeoPandas extends Pandas with a **geometry column** and CRS awareness.
Every row is a feature; the geometry column stores Shapely objects.

```
GeoDataFrame
│
├── column A   (attribute)
├── column B   (attribute)
└── geometry   ← Shapely Point / LineString / Polygon
                  + .crs  ← pyproj.CRS object
```

| Operation | GeoPandas API |
|-----------|--------------|
| Inspect CRS | `gdf.crs` |
| Reproject | `gdf.to_crs(epsg=XXXX)` |
| Area | `gdf.area` (projected CRS only) |
| Buffer | `gdf.buffer(distance)` |
| Spatial join | `gpd.sjoin(left, right)` |
| Plot | `gdf.plot(column="attr")` |

> 💡 The geometry column can be named anything, but **`geometry`** is the convention.
> GeoPandas sets it as the "active geometry" automatically.

In [ ]:
# ── 1a. Creating a GeoDataFrame from scratch ─────────────────────────────────
# A small set of European study cities
cities_data = {
    "id":      [1, 2, 3, 4, 5],
    "name":    ["Amsterdam", "Berlin", "Paris", "Brussels", "Cologne"],
    "country": ["NL", "DE", "FR", "BE", "DE"],
    "pop_m":   [0.87, 3.64, 2.16, 1.18, 1.08],   # population in millions
    "lon":     [4.9041,  13.4050, 2.3522,  4.3517,  6.9603],
    "lat":     [52.3676, 52.5200, 48.8566, 50.8503, 50.9375],
}

# Build geometry column from lon/lat columns
geometry = gpd.points_from_xy(
    cities_data["lon"],
    cities_data["lat"],
    crs="EPSG:4326"          # WGS84
)

cities_gdf = gpd.GeoDataFrame(cities_data, geometry=geometry)

print(f"Type         : {type(cities_gdf)}")
print(f"Shape        : {cities_gdf.shape}")
print(f"CRS          : {cities_gdf.crs}")
print(f"\nFirst row :\n{cities_gdf.iloc[0]}")

In [ ]:
# ── 1b. GeoDataFrame structure and attribute inspection ───────────────────────
print("Column dtypes:")
print(cities_gdf.dtypes)
print()

# The geometry column stores Shapely objects
for row in cities_gdf.itertuples():
    print(f"  {row.name:<12}  geom_type={row.geometry.geom_type}  "
          f"bounds=({row.geometry.x:.4f}, {row.geometry.y:.4f})")

In [ ]:
# ── 1c. Reading GeoJSON from an in-memory dict (same as loading a file) ──────
geojson_dict = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature",
         "geometry": {"type": "Point", "coordinates": [4.9041, 52.3676]},
         "properties": {"name": "Amsterdam", "country": "NL"}},
        {"type": "Feature",
         "geometry": {"type": "Point", "coordinates": [13.4050, 52.5200]},
         "properties": {"name": "Berlin", "country": "DE"}},
    ]
}

# Write the dict to disk so we can use gpd.read_file()
geojson_path = RAW_DIR / "sample_cities.geojson"
geojson_path.write_text(json.dumps(geojson_dict), encoding="utf-8")

loaded_gdf = gpd.read_file(geojson_path)
print("Loaded GDF:")
print(loaded_gdf)
print(f"\nCRS: {loaded_gdf.crs}")

In [ ]:
# ── 1d. Writing a GeoDataFrame to different formats ─────────────────────────
# GeoJSON (text, portable, WGS84 expected)
cities_gdf.to_file(OUT_DIR / "cities.geojson", driver="GeoJSON")

# GeoPackage (modern, multi-layer capable, preferred over Shapefile)
cities_gdf.to_file(OUT_DIR / "cities.gpkg", driver="GPKG")

print("Written:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size:,} bytes)")

### 🎯 Exercise 1 — Explore a GeoDataFrame

**Tasks:**
1. From `cities_gdf`, select only German cities (`country == "DE"`).
2. Print the names and populations of those cities.
3. Add a new column `pop_k` = population in thousands (i.e., `pop_m * 1000`).
4. Print the bounding box of all cities using `cities_gdf.total_bounds`.

```python
# Hint — attribute filter is the same as Pandas:
subset = gdf[gdf["column"] == "value"]
```

In [ ]:
# 🎯 Exercise 1 — your code here ────────────────────────────────────────────

# 1. Select German cities
german_cities = None   # replace with filter

# 2. Print names and populations
# ...

# 3. Add pop_k column
# ...

# 4. Print bounding box
# ...

In [ ]:
# ✅ Exercise 1 — Solution ────────────────────────────────────────────────────

# 1. Select German cities
german_cities = cities_gdf[cities_gdf["country"] == "DE"]
print("1. German cities:")
print(german_cities[["name", "country", "pop_m"]])

# 2. Print names and populations
print()
for _, row in german_cities.iterrows():
    print(f"   {row['name']}: {row['pop_m']} M")

# 3. Add pop_k column
cities_gdf["pop_k"] = (cities_gdf["pop_m"] * 1000).round(0)
print(f"\n3. pop_k column added: {cities_gdf['pop_k'].tolist()}")

# 4. Bounding box
bbox = cities_gdf.total_bounds
print(f"\n4. Bounding box (min_lon, min_lat, max_lon, max_lat):")
print(f"   {bbox}")

---
## 📖 Section 2 — Shapely: Geometry Types and Operations

Shapely implements the OGC Simple Features standard. Every GeoPandas geometry
column cell is a Shapely object.

### Core geometry types

| Type | Example use |
|------|-------------|
| `Point(x, y)` | City centre, GPS fix |
| `LineString([(x,y), ...])` | Road, river, trajectory |
| `Polygon([(x,y), ...])` | Admin boundary, buffer zone |
| `MultiPoint/MultiLineString/MultiPolygon` | Multi-part features |

### Key operations

| Method | Returns |
|--------|---------|
| `.buffer(d)` | Polygon — d units around geometry |
| `.intersection(other)` | Area/line/point of overlap |
| `.union(other)` | Combined geometry |
| `.difference(other)` | Part of self not in other |
| `.centroid` | Centre point |
| `.bounds` | `(min_x, min_y, max_x, max_y)` |
| `.area` | Float (CRS units²) |
| `.length` | Float (CRS units) |

> ⚠️ `.area` and `.length` are in the geometry's CRS units.
> In WGS84 (degrees), those numbers are **not** metres.
> Always project first for metric results.

In [ ]:
# ── 2a. Creating geometry objects ────────────────────────────────────────────
ams   = Point(4.9041, 52.3676)     # Amsterdam
ber   = Point(13.4050, 52.5200)    # Berlin
route = LineString([               # a simplified road corridor
    (4.9041, 52.3676),             # Amsterdam
    (6.9603, 50.9375),             # Cologne
    (9.9937, 53.5507),             # Hamburg
    (13.4050, 52.5200),            # Berlin
])

# The Netherlands bounding polygon (approximate)
nl_poly = Polygon([
    (3.3583, 50.7503),
    (7.2275, 50.7503),
    (7.2275, 53.5557),
    (3.3583, 53.5557),
    (3.3583, 50.7503),          # close the ring
])

print(f"Amsterdam  : {ams}")
print(f"Route      : {route.geom_type}, {len(route.coords)} vertices")
print(f"NL polygon : {nl_poly.geom_type}, area≈{nl_poly.area:.4f} deg²")

In [ ]:
# ── 2b. Geometric properties ─────────────────────────────────────────────────
print("Route properties:")
print(f"  length (degrees)  : {route.length:.4f}")
print(f"  bounds            : {route.bounds}")
print(f"  centroid          : {route.centroid}")

print("\nNL polygon properties:")
print(f"  area (degrees²)   : {nl_poly.area:.4f}")
print(f"  centroid          : {nl_poly.centroid}")
print(f"  is_valid          : {nl_poly.is_valid}")
print(f"  is_simple         : {nl_poly.is_simple}")

In [ ]:
# ── 2c. Topology predicates ───────────────────────────────────────────────────
print("Topology predicates:")
print(f"  ams within nl_poly       : {ams.within(nl_poly)}")
print(f"  ber within nl_poly       : {ber.within(nl_poly)}")
print(f"  route intersects nl_poly : {route.intersects(nl_poly)}")
print(f"  route crosses nl_poly    : {route.crosses(nl_poly)}")

# The intersection returns the actual crossing geometry
crossing = route.intersection(nl_poly)
print(f"\nRoute x NL crossing: {crossing.geom_type}")

In [ ]:
# ── 2d. Buffer, union, and difference ────────────────────────────────────────
# Small buffer around Amsterdam in degree units (≈ 0.5° ~ 55 km at this lat)
ams_buffer = ams.buffer(0.5)
ber_buffer = ber.buffer(0.5)

print(f"Amsterdam buffer area  : {ams_buffer.area:.4f} deg²")
print(f"Berlin buffer area     : {ber_buffer.area:.4f} deg²")

# Union — combined footprint
combined = ams_buffer.union(ber_buffer)
print(f"Union geom_type        : {combined.geom_type}")
print(f"Union area             : {combined.area:.4f} deg²")

# Difference — ams buffer minus nl_poly
ams_outside_nl = ams_buffer.difference(nl_poly)
print(f"\nAms buffer outside NL  : {ams_outside_nl.geom_type}")
print(f"  Area outside NL      : {ams_outside_nl.area:.4f} deg²")

### 🎯 Exercise 2 — Shapely Operations

Use the geometries already defined (`ams`, `ber`, `nl_poly`, `route`).

**Tasks:**
1. Create a Polygon representing Germany's rough bounding box:
   `lon 5.9–15.0 | lat 47.3–55.1`
2. Check whether Berlin (`ber`) is **within** the Germany polygon.
3. Compute the **intersection** of `nl_poly` and the Germany polygon.
4. Print the geometry type and area of the intersection.

```python
# Hint — Polygon takes a list of (x, y) tuples forming a closed ring
de_bbox = Polygon([...])
```

In [ ]:
# 🎯 Exercise 2 — your code here ────────────────────────────────────────────

# 1. Germany bounding polygon
de_bbox = None   # replace

# 2. Berlin within Germany?
# ...

# 3. Intersection of NL and DE
nl_de_intersection = None   # replace

# 4. Print type and area
# ...

In [ ]:
# ✅ Exercise 2 — Solution ────────────────────────────────────────────────────

# 1. Germany bounding polygon
de_bbox = Polygon([
    (5.9, 47.3), (15.0, 47.3), (15.0, 55.1), (5.9, 55.1), (5.9, 47.3)
])
print(f"1. DE bbox valid: {de_bbox.is_valid}")

# 2. Berlin within Germany?
print(f"2. Berlin within DE bbox: {ber.within(de_bbox)}")

# 3. Intersection of NL and DE (border zone)
nl_de_intersection = nl_poly.intersection(de_bbox)
print(f"3. Intersection type : {nl_de_intersection.geom_type}")
print(f"4. Intersection area : {nl_de_intersection.area:.4f} deg²")

---
## 📖 Section 3 — PyProj: CRS and Coordinate Transformations

Covered in Week 4 theory — here we apply PyProj directly inside GeoPandas workflows.

| Scenario | API |
|----------|-----|
| Inspect GDF CRS | `gdf.crs` → `pyproj.CRS` |
| Reproject whole GDF | `gdf.to_crs(epsg=XXXX)` |
| Transform individual coordinates | `Transformer.from_crs(...).transform(x, y)` |
| Geodesic distance | `Geod(ellps="WGS84").inv(lon1, lat1, lon2, lat2)` |

### Common EPSG codes for Western Europe

| EPSG | Name | Units | Use case |
|------|------|-------|---------|
| 4326 | WGS84 | degrees | storage, interchange |
| 3857 | Web Mercator | metres | web tiles (distorted area) |
| 32631 | UTM zone 31N | metres | NL/BE metric analysis |
| 32632 | UTM zone 32N | metres | DE/DK metric analysis |
| 25832 | ETRS89 / UTM 32N | metres | European standard |

> ⚠️ Use `always_xy=True` in `Transformer.from_crs()`.
> Without it, older EPSG definitions with lat/lon axis order will swap your coordinates.

In [ ]:
# ── 3a. CRS inspection and comparison ────────────────────────────────────────
crs_wgs84  = CRS.from_epsg(4326)
crs_utm31n = CRS.from_epsg(32631)

print("WGS84:")
print(f"  name          : {crs_wgs84.name}")
print(f"  is_geographic : {crs_wgs84.is_geographic}")
print(f"  axis units    : {[ax.unit_name for ax in crs_wgs84.axis_info]}")

print("\nUTM 31N (EPSG:32631):")
print(f"  name          : {crs_utm31n.name}")
print(f"  is_projected  : {crs_utm31n.is_projected}")
print(f"  axis units    : {[ax.unit_name for ax in crs_utm31n.axis_info]}")

In [ ]:
# ── 3b. Reprojecting a GeoDataFrame ──────────────────────────────────────────
print("Before reproject:")
print(f"  CRS   : {cities_gdf.crs}")
print(f"  First point: {cities_gdf.geometry.iloc[0]}")

# to_crs returns a NEW GeoDataFrame — original unchanged
cities_utm = cities_gdf.to_crs(epsg=32631)

print("\nAfter reproject to EPSG:32631 (UTM 31N):")
print(f"  CRS   : {cities_utm.crs}")
print(f"  First point: {cities_utm.geometry.iloc[0].coords[0]}")
print("  (coordinates now in metres, not degrees)")

In [ ]:
# ── 3c. Geodesic distance with PyProj Geod ───────────────────────────────────
geod = Geod(ellps="WGS84")

ams_row  = cities_gdf[cities_gdf["name"] == "Amsterdam"].iloc[0]
ber_row  = cities_gdf[cities_gdf["name"] == "Berlin"].iloc[0]
par_row  = cities_gdf[cities_gdf["name"] == "Paris"].iloc[0]

_, _, ams_ber_m = geod.inv(ams_row.geometry.x, ams_row.geometry.y,
                           ber_row.geometry.x, ber_row.geometry.y)
_, _, ams_par_m = geod.inv(ams_row.geometry.x, ams_row.geometry.y,
                           par_row.geometry.x, par_row.geometry.y)

print(f"Amsterdam → Berlin : {ams_ber_m/1000:.1f} km (geodesic)")
print(f"Amsterdam → Paris  : {ams_par_m/1000:.1f} km (geodesic)")

### 🎯 Exercise 3 — Reproject and Compare

**Tasks:**
1. Reproject `cities_gdf` to **EPSG:3857** (Web Mercator) and store as `cities_3857`.
2. Print the x-coordinate (easting) of Amsterdam in both EPSG:32631 and EPSG:3857.
3. Compute the Euclidean distance (metres) between Amsterdam and Berlin in UTM 31N.
4. Compare with the geodesic distance printed in cell 3c above.

```python
# Hint — Euclidean distance between two Shapely Points:
import math
d = math.dist(point_a.coords[0], point_b.coords[0])
```

In [ ]:
# 🎯 Exercise 3 — your code here ────────────────────────────────────────────

# 1. Reproject to Web Mercator
cities_3857 = None   # replace

# 2. Amsterdam easting in 32631 vs 3857
# ...

# 3. Euclidean distance in UTM 31N
# ...

# 4. Comment comparing with geodesic distance
# ...

In [ ]:
# ✅ Exercise 3 — Solution ────────────────────────────────────────────────────
import math

# 1.
cities_3857 = cities_gdf.to_crs(epsg=3857)
print(f"1. cities_3857 CRS: {cities_3857.crs}")

# 2.
ams_utm = cities_utm[cities_utm["name"] == "Amsterdam"].iloc[0].geometry
ams_web = cities_3857[cities_3857["name"] == "Amsterdam"].iloc[0].geometry
print(f"\n2. Amsterdam easting UTM31N  : {ams_utm.x:,.0f} m")
print(f"   Amsterdam easting 3857    : {ams_web.x:,.0f} m")

# 3.
ber_utm = cities_utm[cities_utm["name"] == "Berlin"].iloc[0].geometry
eucl_m  = math.dist(ams_utm.coords[0], ber_utm.coords[0])
print(f"\n3. Euclidean (UTM 31N) Ams→Ber: {eucl_m/1000:.1f} km")
print(f"   Geodesic               Ams→Ber: {ams_ber_m/1000:.1f} km")
print(f"   Difference: {abs(eucl_m-ams_ber_m):.0f} m")
print("   (small diff because both cities sit near UTM 31N central meridian)")

---
## 📖 Section 4 — Spatial Measurements: Distance, Area, Length

A cardinal rule:

> **Always project to a metric CRS before measuring area, length, or distance.**

| CRS | `.area` units | `.length` units |
|-----|--------------|----------------|
| EPSG:4326 (WGS84) | degrees² — meaningless | degrees — meaningless |
| EPSG:32631 (UTM 31N) | m² | m |
| EPSG:3857 (Web Mercator) | m² (distorted at poles) | m (distorted) |

Use **UTM zones** or national grids for high-accuracy local work.
Use **geodesic methods** (`Geod.inv`) for cross-region work.

In [ ]:
# ── 4a. Buffer in metric CRS then measure area ────────────────────────────────
# Buffer 50 km around each city
cities_utm["buffer_50km"] = cities_utm.geometry.buffer(50_000)   # 50 000 m

# The geometry column is still Points — set buffer as active geom to measure area
buf_gdf = cities_utm.copy()
buf_gdf = buf_gdf.set_geometry("buffer_50km")

print("50 km buffer areas:")
for _, row in buf_gdf.iterrows():
    area_km2 = row.geometry.area / 1e6
    print(f"  {row['name']:<12}  {area_km2:,.1f} km²  (expected ≈ {math.pi*50**2:.0f} km²)")

In [ ]:
# ── 4b. Length of a LineString in projected CRS ───────────────────────────────
# Our Amsterdam → Cologne → Hamburg → Berlin route
route_wgs84 = LineString([
    (4.9041, 52.3676),   # Amsterdam
    (6.9603, 50.9375),   # Cologne
    (9.9937, 53.5507),   # Hamburg
    (13.4050, 52.5200),  # Berlin
])

# Transform to UTM 31N using pyproj
t = Transformer.from_crs(4326, 32631, always_xy=True)
coords_utm = [t.transform(x, y) for x, y in route_wgs84.coords]
route_utm  = LineString(coords_utm)

print(f"Route length (degrees) : {route_wgs84.length:.4f}  ← meaningless")
print(f"Route length (UTM 31N) : {route_utm.length/1000:.1f} km")

# Compare with geodesic sum of segments
geod = Geod(ellps="WGS84")
seg_total = 0.0
coords = list(route_wgs84.coords)
for (x1, y1), (x2, y2) in zip(coords[:-1], coords[1:]):
    _, _, d = geod.inv(x1, y1, x2, y2)
    seg_total += d

print(f"Route length (geodesic): {seg_total/1000:.1f} km")

In [ ]:
# ── 4c. Distance matrix between cities ───────────────────────────────────────
names  = cities_gdf["name"].tolist()
geoms  = cities_gdf["geometry"].tolist()

print("Geodesic distance matrix (km):")
header = f"{'':>12}" + "".join(f"{n[:5]:>8}" for n in names)
print(header)

for i, (n1, g1) in enumerate(zip(names, geoms)):
    row = f"{n1[:12]:>12}"
    for j, (n2, g2) in enumerate(zip(names, geoms)):
        if i == j:
            row += f"{'—':>8}"
        else:
            _, _, d = geod.inv(g1.x, g1.y, g2.x, g2.y)
            row += f"{d/1000:>8.0f}"
    print(row)

### 🎯 Exercise 4 — Area comparison

**Tasks:**
1. Create a 100 km buffer around Amsterdam in UTM 31N.
2. Compute its area in km².
3. Create the same buffer in **EPSG:3857** (Web Mercator).
4. Compute its area in km² and compare the two results.
5. Explain in a comment why the two areas differ slightly.

```python
# Hint — area in km² = geom.area / 1_000_000
```

In [ ]:
# 🎯 Exercise 4 — your code here ────────────────────────────────────────────

# 1. 100 km buffer in UTM 31N
ams_utm_pt = cities_utm[cities_utm["name"] == "Amsterdam"].iloc[0].geometry
buf_utm     = None   # replace

# 2. Area in km²
# ...

# 3. 100 km buffer in EPSG:3857
ams_3857_pt = cities_3857[cities_3857["name"] == "Amsterdam"].iloc[0].geometry
buf_3857    = None   # replace

# 4. Area in km²
# ...

# 5. Explanation comment
# ...

In [ ]:
# ✅ Exercise 4 — Solution ────────────────────────────────────────────────────

# 1.
ams_utm_pt = cities_utm[cities_utm["name"] == "Amsterdam"].iloc[0].geometry
buf_utm    = ams_utm_pt.buffer(100_000)

# 2.
area_utm_km2 = buf_utm.area / 1e6
print(f"1-2. UTM 31N  100km buffer area : {area_utm_km2:,.1f} km²")
print(f"     Expected (π r²)            : {math.pi*100**2:,.1f} km²")

# 3.
ams_3857_pt = cities_3857[cities_3857["name"] == "Amsterdam"].iloc[0].geometry
buf_3857    = ams_3857_pt.buffer(100_000)

# 4.
area_3857_km2 = buf_3857.area / 1e6
print(f"\n3-4. 3857     100km buffer area : {area_3857_km2:,.1f} km²")
print(f"     Difference                 : {abs(area_utm_km2-area_3857_km2):,.1f} km²")

# 5.
print('''
5. Web Mercator (3857) is a cylindrical conformal projection that preserves shape
   locally but distorts area, especially at latitudes away from the equator.
   At 52°N (Amsterdam), the scale factor > 1, so distances and buffers in 3857
   are accurate in shape but slightly off in size compared to UTM (equal-area
   at the UTM central meridian). For high-precision area work, always use a
   local equal-area or UTM projection.''')

---
## 📖 Section 5 — Spatial Joins and Overlays

### Spatial join (gpd.sjoin)

Joins attributes from one layer to another based on spatial relationship.

```python
gpd.sjoin(left_gdf, right_gdf, how="left", predicate="within")
```

| `predicate` | Meaning |
|-------------|---------|
| `"within"` | left geometry is inside right |
| `"contains"` | left geometry contains right |
| `"intersects"` | geometries overlap (default) |

### Overlay (gpd.overlay)

Creates new geometries from the overlap of two polygon layers.

```python
gpd.overlay(gdf1, gdf2, how="intersection")
```

| `how` | Result |
|-------|--------|
| `"intersection"` | only the overlapping area |
| `"union"` | everything from both layers |
| `"difference"` | gdf1 minus overlap |
| `"symmetric_difference"` | everything except the overlap |

In [ ]:
# ── 5a. Create sample polygon layers (admin zones) ───────────────────────────
# Two fictional administrative regions around our study area
zone_a = Polygon([
    (3.5, 50.5), (7.5, 50.5), (7.5, 53.5), (3.5, 53.5), (3.5, 50.5)
])
zone_b = Polygon([
    (6.5, 49.5), (14.5, 49.5), (14.5, 54.0), (6.5, 54.0), (6.5, 49.5)
])

zones_gdf = gpd.GeoDataFrame(
    {"zone_id": ["A", "B"], "zone_name": ["West Region", "East Region"]},
    geometry=[zone_a, zone_b],
    crs="EPSG:4326"
)

print("Zones:")
print(zones_gdf[["zone_id", "zone_name"]])

In [ ]:
# ── 5b. Spatial join — point-in-polygon ──────────────────────────────────────
# Which zone does each city fall in?
cities_in_zones = gpd.sjoin(
    cities_gdf,
    zones_gdf[["zone_id", "zone_name", "geometry"]],
    how="left",
    predicate="within"
)

print("Cities with zone assignment:")
cols = ["name", "country", "zone_id", "zone_name"]
print(cities_in_zones[cols].to_string(index=False))
print(f"\n(NaN means the city is not within any single zone)") 

In [ ]:
# ── 5c. Overlay — intersection of two polygon layers ─────────────────────────
overlap = gpd.overlay(
    zones_gdf[["zone_id", "geometry"]].rename(columns={"zone_id": "zone_A"}),
    zones_gdf[["zone_id", "geometry"]].rename(columns={"zone_id": "zone_B"}),
    how="intersection"
)

# Self-overlay returns each pair — filter to only A∩B
cross = overlap[overlap["zone_A"] != overlap["zone_B"]]

if len(cross) > 0:
    area_deg2 = cross.iloc[0].geometry.area
    print(f"Overlap between zones A and B:")
    print(f"  area   : {area_deg2:.4f} deg²")
    print(f"  bounds : {cross.iloc[0].geometry.bounds}")
else:
    print("Zones do not overlap.")

### 🎯 Exercise 5 — Spatial Join with Buffer Zones

**Tasks:**
1. Buffer `cities_utm` by 80 km.
2. Create a GeoDataFrame of buffered cities (call it `city_buffers_gdf`).
3. Use `gpd.sjoin` to find which cities' 80 km buffers **intersect** with each other.
4. Print the unique pairs.

```python
# Hint — create a GDF from the buffered geometry column:
city_buffers_gdf = cities_utm.copy()
city_buffers_gdf.geometry = cities_utm.geometry.buffer(80_000)
```

In [ ]:
# 🎯 Exercise 5 — your code here ────────────────────────────────────────────

# 1-2. Buffer cities and create GDF
city_buffers_gdf = None   # replace

# 3. sjoin — which buffers intersect each other?
overlapping = None   # replace

# 4. Print unique pairs
# ...

In [ ]:
# ✅ Exercise 5 — Solution ────────────────────────────────────────────────────

# 1-2.
city_buffers_gdf = cities_utm.copy()
city_buffers_gdf = city_buffers_gdf.set_geometry(cities_utm.geometry.buffer(80_000))

# 3. Self-join on intersection
overlapping = gpd.sjoin(
    city_buffers_gdf[["name", "geometry"]],
    city_buffers_gdf[["name", "geometry"]],
    how="left",
    predicate="intersects"
)

# 4. Keep only cross-city pairs (not self)
pairs = overlapping[overlapping["name_left"] < overlapping["name_right"]][
    ["name_left", "name_right"]
].drop_duplicates()
print("Cities whose 80 km buffers intersect:")
print(pairs.to_string(index=False))

---
## 📖 Section 6 — Fiona: Reading and Writing Vector Files

Fiona is the low-level OGR-based driver that powers `gpd.read_file()`.
Using it directly gives you feature-by-feature control — useful for large files,
streaming, or custom property mapping.

| Use case | Fiona or GeoPandas? |
|----------|---------------------|
| Quick analysis | `gpd.read_file()` — easier |
| Large files (stream row by row) | `fiona.open()` directly |
| Custom schema / projection on write | `fiona.open()` directly |
| Format inspection (layers, schema) | `fiona.listlayers()`, `fiona.open()` |

In [ ]:
# ── 6a. Inspect file metadata with Fiona ──────────────────────────────────────
geojson_file = OUT_DIR / "cities.geojson"

with fiona.open(geojson_file) as src:
    print(f"Driver   : {src.driver}")
    print(f"CRS      : {src.crs}")
    print(f"Features : {len(src)}")
    print(f"Schema   : {src.schema}")
    print(f"Bounds   : {src.bounds}")

In [ ]:
# ── 6b. Reading features one-by-one with Fiona ───────────────────────────────
with fiona.open(geojson_file) as src:
    for i, feat in enumerate(src):
        props = feat["properties"]
        geom  = feat["geometry"]
        print(f"[{i}] {props['name']:<12}  lon={geom['coordinates'][0]:.4f}  "
              f"lat={geom['coordinates'][1]:.4f}")

In [ ]:
# ── 6c. Writing a new file with Fiona directly ───────────────────────────────
fiona_out = OUT_DIR / "cities_fiona.gpkg"

# Define the schema
schema = {
    "geometry": "Point",
    "properties": {
        "id":      "int",
        "name":    "str",
        "country": "str",
        "pop_m":   "float",
    }
}

with fiona.open(fiona_out, "w",
                driver="GPKG",
                schema=schema,
                crs="EPSG:4326") as dst:
    for _, row in cities_gdf.iterrows():
        dst.write({
            "geometry": mapping(row.geometry),
            "properties": {
                "id":      int(row["id"]),
                "name":    row["name"],
                "country": row["country"],
                "pop_m":   float(row["pop_m"]),
            }
        })

print(f"Written: {fiona_out} ({fiona_out.stat().st_size:,} bytes)")

# Read back to verify
with fiona.open(fiona_out) as src:
    print(f"Features written: {len(src)}")
    print(f"Schema: {src.schema}")

### 🎯 Exercise 6 — Filter on Read with Fiona

**Tasks:**
1. Open `cities.geojson` with `fiona.open()`.
2. Read only features where `country == "DE"`.
3. Collect them in a list and print the names.

```python
# Hint — Fiona features are dict-like:
feat["properties"]["country"]
```

In [ ]:
# 🎯 Exercise 6 — your code here ────────────────────────────────────────────

german_feats = []
# TODO: open geojson_file and filter to DE features

print("German features:", german_feats)

In [ ]:
# ✅ Exercise 6 — Solution ────────────────────────────────────────────────────
german_feats = []
with fiona.open(geojson_file) as src:
    for feat in src:
        if feat["properties"].get("country") == "DE":
            german_feats.append(feat["properties"]["name"])

print("German cities found via Fiona:", german_feats)

---
## 📖 Section 7 — Rasterio: Raster Data Introduction

Rasters store spatially referenced grids of values (elevation, temperature,
satellite imagery, population density …).

```
Raster file
├── bands       ← 1-N arrays of values (e.g. R, G, B or elevation)
├── transform   ← maps pixel → real-world coordinate (affine matrix)
├── CRS         ← spatial reference of the grid
└── nodata      ← value used for "no data" cells
```

| Task | API |
|------|-----|
| Open a file | `rasterio.open(path)` |
| Read a band | `src.read(1)` → numpy 2-D array |
| Get transform | `src.transform` |
| Get pixel coords | `src.xy(row, col)` |
| Get CRS | `src.crs` |
| Write a raster | `rasterio.open(path, "w", ...)` |

> 💡 Week 5 gives you a working introduction.
> Rasterio is covered more deeply in Module C of the full course.

In [ ]:
# ── 7a. Create a synthetic mini-raster (elevation grid) ──────────────────────
import numpy as np

# A 10×10 "elevation" grid over Amsterdam region (3–6°E, 51–54°N)
nrows, ncols = 10, 10
west, south, east, north = 3.0, 51.0, 6.0, 54.0

# Simulate elevation: higher in the east (just for demo)
np.random.seed(42)
elevation = (np.random.rand(nrows, ncols) * 50 + 0).astype("float32")
# Add a fake ridge
elevation[:, 6:] += 30

transform = from_bounds(west, south, east, north, ncols, nrows)
raster_path = OUT_DIR / "dem_sample.tif"

with rasterio.open(
    raster_path, "w",
    driver="GTiff",
    height=nrows, width=ncols,
    count=1,
    dtype=elevation.dtype,
    crs="EPSG:4326",
    transform=transform,
    nodata=-9999.0
) as dst:
    dst.write(elevation, 1)

print(f"Written: {raster_path} ({raster_path.stat().st_size:,} bytes)")

In [ ]:
# ── 7b. Reading raster metadata and data ─────────────────────────────────────
with rasterio.open(raster_path) as src:
    print(f"Driver      : {src.driver}")
    print(f"CRS         : {src.crs}")
    print(f"Shape       : {src.height} rows × {src.width} cols")
    print(f"Bands       : {src.count}")
    print(f"Dtype       : {src.dtypes[0]}")
    print(f"Nodata      : {src.nodata}")
    print(f"Transform   : {src.transform}")

    band1 = src.read(1)          # 2-D numpy array
    print(f"\nBand 1 shape   : {band1.shape}")
    print(f"Min elevation  : {band1.min():.2f}")
    print(f"Max elevation  : {band1.max():.2f}")
    print(f"Mean elevation : {band1.mean():.2f}")

In [ ]:
# ── 7c. Sampling raster values at city locations ─────────────────────────────
from rasterio.sample import sample_gen

# City coordinates: list of (lon, lat) tuples
city_coords = list(zip(cities_gdf["lon"], cities_gdf["lat"]))

with rasterio.open(raster_path) as src:
    sampled = list(sample_gen(src, city_coords))

print("Sampled elevation values at city locations:")
for city, (lon, lat), (val,) in zip(
    cities_gdf["name"], city_coords, sampled
):
    print(f"  {city:<12}  lon={lon:.2f} lat={lat:.2f}  elev={val:.1f} m")

### 🎯 Exercise 7 — Raster Statistics

**Tasks:**
1. Open `dem_sample.tif` and read band 1 into a numpy array.
2. Compute and print: min, max, mean, standard deviation.
3. Count how many pixels have elevation **above 40 m**.
4. Print that count and the percentage of total pixels.

```python
# Hint — numpy boolean indexing:
high_pixels = arr[arr > threshold]
```

In [ ]:
# 🎯 Exercise 7 — your code here ────────────────────────────────────────────

# 1. Open and read band 1
with rasterio.open(raster_path) as src:
    arr = None   # replace with src.read(...)

# 2. Statistics
# ...

# 3-4. Pixels above 40 m
# ...

In [ ]:
# ✅ Exercise 7 — Solution ────────────────────────────────────────────────────

# 1.
with rasterio.open(raster_path) as src:
    arr = src.read(1)

# 2.
print("2. Raster statistics:")
print(f"   Min  : {arr.min():.2f} m")
print(f"   Max  : {arr.max():.2f} m")
print(f"   Mean : {arr.mean():.2f} m")
print(f"   Std  : {arr.std():.2f} m")

# 3-4.
threshold   = 40.0
above_40    = (arr > threshold).sum()
total_pix   = arr.size
pct         = above_40 / total_pix * 100
print(f"\n3-4. Pixels above 40 m : {above_40} / {total_pix} ({pct:.1f}%)")

---
## 📖 Section 8 — Visualization: Matplotlib and Folium

GeoPandas provides `.plot()` built on **Matplotlib** for static maps.
**Folium** creates interactive Leaflet.js maps from Python.

| Tool | Best for |
|------|---------|
| `gdf.plot()` | Quick static map, publication figures |
| `folium.Map()` | Interactive web map, user-facing dashboards |
| Combining both | Export static + embed interactive in notebook |

> 💡 In JupyterLab and Colab, `folium` maps render inline.
> Outside Jupyter, call `m.save("map.html")` and open in a browser.

In [ ]:
# ── 8a. Static map — cities + zone polygons ──────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))

# Background zones
zones_gdf.plot(ax=ax, color="lightyellow", edgecolor="#aaa", linewidth=1, alpha=0.7)

# City buffers (80 km) in UTM, back-projected for display
buf_disp = gpd.GeoDataFrame(
    cities_utm[["name"]].copy(),
    geometry=cities_utm.geometry.buffer(80_000),
    crs=cities_utm.crs
).to_crs(epsg=4326)
buf_disp.plot(ax=ax, color="steelblue", alpha=0.15, edgecolor="steelblue", linewidth=0.5)

# City points sized by population
cities_gdf.plot(
    ax=ax, column="pop_m",
    markersize=cities_gdf["pop_m"] * 30,
    cmap="Reds", legend=True,
    legend_kwds={"label": "Population (millions)", "shrink": 0.5}
)

# City labels
for _, row in cities_gdf.iterrows():
    ax.annotate(
        row["name"],
        xy=(row.geometry.x, row.geometry.y),
        xytext=(3, 5), textcoords="offset points",
        fontsize=8, color="#222"
    )

ax.set_title("Week 5 Study Area — Cities, Zones, and 80 km Buffers", pad=14)
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig(OUT_DIR / "week5_static_map.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: week5_static_map.png")

In [ ]:
# ── 8b. Interactive Folium map ────────────────────────────────────────────────
centre_lat = cities_gdf["lat"].mean()
centre_lon = cities_gdf["lon"].mean()

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=6,
               tiles="CartoDB positron")

# Add zone polygons
for _, row in zones_gdf.iterrows():
    folium.GeoJson(
        row.geometry.__geo_interface__,
        name=row["zone_name"],
        style_function=lambda _: {
            "fillColor": "lightyellow", "color": "orange",
            "weight": 1.5, "fillOpacity": 0.3
        },
        tooltip=row["zone_name"]
    ).add_to(m)

# Add city markers
colour_map = {"NL": "blue", "DE": "red", "FR": "purple", "BE": "green"}
for _, row in cities_gdf.iterrows():
    popup_html = (
        f"<b>{row['name']}</b><br>"
        f"Country : {row['country']}<br>"
        f"Pop     : {row['pop_m']} M<br>"
        f"Lon/Lat : {row.geometry.x:.4f}, {row.geometry.y:.4f}"
    )
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=max(6, row["pop_m"] * 3),
        color=colour_map.get(row["country"], "gray"),
        fill=True, fill_opacity=0.75,
        popup=folium.Popup(popup_html, max_width=200),
        tooltip=row["name"]
    ).add_to(m)

folium.LayerControl().add_to(m)
m.save(str(OUT_DIR / "week5_interactive_map.html"))
print("Saved: week5_interactive_map.html")
try:
    display(m)
except NameError:
    print("Open week5_interactive_map.html in a browser to view.")

### 🎯 Exercise 8 — Themed Choropleth

**Tasks:**
1. Add a `density` column to `cities_gdf`: `pop_m / (π × (50km/111km)²)`
   (rough population density in the 50 km radius, where 1° ≈ 111 km).
2. Plot a **choropleth** of the `density` column using `gdf.plot(column="density", cmap="YlOrRd")`.
3. Add a title and legend.

```python
# Hint — 50 km in degrees ≈ 50 / 111
```

In [ ]:
# 🎯 Exercise 8 — your code here ────────────────────────────────────────────

# 1. Add density column
cities_gdf["density"] = None   # replace

# 2-3. Choropleth plot
# ...

In [ ]:
# ✅ Exercise 8 — Solution ────────────────────────────────────────────────────

# 1.
radius_deg = 50 / 111
area_deg2  = math.pi * radius_deg ** 2
cities_gdf["density"] = (cities_gdf["pop_m"] / area_deg2).round(2)
print("1. Density column (M pop / deg²):")
print(cities_gdf[["name", "pop_m", "density"]].to_string(index=False))

# 2-3.
fig, ax = plt.subplots(figsize=(8, 6))
zones_gdf.plot(ax=ax, color="whitesmoke", edgecolor="#ccc", linewidth=0.8)
cities_gdf.plot(
    ax=ax, column="density", cmap="YlOrRd",
    markersize=80, legend=True,
    legend_kwds={"label": "Pop density (M/deg²)", "shrink": 0.5}
)
for _, row in cities_gdf.iterrows():
    ax.annotate(row["name"], xy=(row.geometry.x, row.geometry.y),
                xytext=(3, 5), textcoords="offset points", fontsize=8)
ax.set_title("City Population Density — 50 km radius")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

---
## 🔬 Mini-Lab: Local Study Area Analysis

### Scenario
You work for an urban analytics consultancy. A client wants to know:
1. Which cities in the study area lie within their two administrative zones?
2. What is the population **density** (per km²) within a 100 km radius of each city?
3. Are there significant overlaps between the 100 km service areas?
4. Produce a summary report (CSV) and an interactive map.

### Setup — Combined dataset
Run the cells in order. Every output goes into `data/week_05/output/`.

### Logging

In [ ]:
# 🔬 Mini-Lab — configure logging
log_path = LOGS_DIR / "week5_lab.log"
logger = logging.getLogger("week5.lab")
logger.setLevel(logging.INFO)
logger.handlers.clear()
fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
for h in [logging.FileHandler(log_path, encoding="utf-8"),
          logging.StreamHandler()]:
    h.setFormatter(fmt)
    logger.addHandler(h)
logger.info("Mini-Lab started")

In [ ]:
# 🔬 Mini-Lab — Part A: zone assignment ─────────────────────────────────────
logger.info("Part A: spatial join — cities to zones")

cities_with_zones = gpd.sjoin(
    cities_gdf[["name", "country", "pop_m", "geometry"]],
    zones_gdf[["zone_id", "zone_name", "geometry"]],
    how="left",
    predicate="within"
)

print("Zone assignment:")
print(cities_with_zones[["name", "zone_id", "zone_name"]].to_string(index=False))
logger.info("Zone assignment complete: %d cities", len(cities_with_zones))

In [ ]:
# 🔬 Mini-Lab — Part B: 100 km service areas ────────────────────────────────
logger.info("Part B: 100 km buffers")

service_areas = cities_utm.copy()
service_areas.geometry = cities_utm.geometry.buffer(100_000)   # 100 km
service_areas_wgs = service_areas.to_crs(epsg=4326)

print("Service areas:")
for _, row in service_areas.iterrows():
    area_km2 = row.geometry.area / 1e6
    print(f"  {row['name']:<12}  area = {area_km2:,.0f} km²")
    logger.info("  %s service area: %.0f km²", row["name"], area_km2)

In [ ]:
# �� Mini-Lab — Part C: overlapping service areas ────────────────────────────
logger.info("Part C: find overlapping service areas")

overlap_join = gpd.sjoin(
    service_areas[["name", "geometry"]],
    service_areas[["name", "geometry"]],
    how="left",
    predicate="intersects"
)
overlaps = overlap_join[overlap_join["name_left"] < overlap_join["name_right"]]                [["name_left", "name_right"]].drop_duplicates()

print("Service area overlaps:")
for _, row in overlaps.iterrows():
    print(f"  {row['name_left']} ↔ {row['name_right']}")
    logger.info("Overlap: %s ↔ %s", row["name_left"], row["name_right"])

In [ ]:
# 🔬 Mini-Lab — Part D: summary report CSV ──────────────────────────────────
logger.info("Part D: write summary report")

import csv

# Merge zone info into service areas dataframe
summary_data = cities_gdf.merge(
    cities_with_zones[["name", "zone_id", "zone_name"]].drop_duplicates("name"),
    on="name", how="left"
)

# Compute geodesic distances to Brussels (reference point)
bru_lon, bru_lat = 4.3517, 50.8503
summary_data["dist_bru_km"] = summary_data.apply(
    lambda row: geod.inv(row.geometry.x, row.geometry.y, bru_lon, bru_lat)[2] / 1000,
    axis=1
).round(1)

report_path = OUT_DIR / "week5_summary.csv"
summary_data[["name", "country", "pop_m", "zone_id", "dist_bru_km"]].to_csv(
    report_path, index=False
)
print(f"Saved: {report_path}")
print(report_path.read_text(encoding="utf-8"))
logger.info("Summary CSV written: %s", report_path)

In [ ]:
# 🔬 Mini-Lab — Final map ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

# Zones background
zones_gdf.plot(ax=ax, color="lightyellow", edgecolor="orange",
               linewidth=1.5, alpha=0.6, label="Admin zones")

# Service areas
service_areas_wgs.plot(ax=ax, color="steelblue", alpha=0.12,
                       edgecolor="steelblue", linewidth=0.6)

# City points
cities_gdf.plot(ax=ax, color="darkred", markersize=50, zorder=5)

# Labels
for _, row in cities_gdf.iterrows():
    ax.annotate(
        f"{row['name']}\n{row['pop_m']}M",
        xy=(row.geometry.x, row.geometry.y),
        xytext=(4, 5), textcoords="offset points",
        fontsize=8, color="#111",
        bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.6)
    )

handles = [
    mpatches.Patch(color="lightyellow", label="Admin zones"),
    mpatches.Patch(color="steelblue", alpha=0.4, label="100 km service areas"),
    plt.scatter([], [], c="darkred", s=50, label="Cities"),
]
ax.legend(handles=handles, loc="lower right", fontsize=9)
ax.set_title("Week 5 Mini-Lab — Study Area Overview", fontsize=13, pad=14)
ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.grid(True, linestyle="--", alpha=0.35)
plt.tight_layout()
plt.savefig(OUT_DIR / "week5_minilab_map.png", dpi=130, bbox_inches="tight")
plt.show()
logger.info("Mini-Lab complete")
print("\nLog file:")
print(log_path.read_text(encoding="utf-8"))

### 🚀 Extension Exercises (optional, not graded)

**A. Nearest-neighbour join**
Add a sixth city (e.g. Copenhagen at lon=12.57, lat=55.68) and find the **nearest**
existing city to it using `gpd.sjoin_nearest()`.

**B. Raster + vector overlay**
Clip the `dem_sample.tif` to the Netherlands bounding polygon (`nl_poly`)
using `rasterio.mask.mask()`. Print min and max elevation within the clip.

**C. Choropleth by zone**
Dissolve `cities_with_zones` by `zone_id` to produce two rows,
summing population per zone, then produce a polygon choropleth.

**D. Export to multiple formats**
Write `cities_gdf` to Shapefile, GeoPackage, and GeoJSON.
Read each back and assert that feature counts match.

**E. Track your pipeline**
Wrap the Mini-Lab Parts A–D in a function `run_lab(input_gdf, zones_gdf)`
that returns a summary dict and accepts a logger parameter.

---
## ✅ Week 5 Summary

Congratulations on completing Week 5!

| Section | Key concepts mastered |
|---------|-----------------------|
| **1. GeoPandas** | `GeoDataFrame`, `read_file`, `to_file`, `.crs`, `.to_crs()`, `.total_bounds` |
| **2. Shapely** | `Point`, `LineString`, `Polygon`, `.buffer()`, `.intersection()`, `.union()`, topology predicates |
| **3. PyProj** | `CRS`, `Transformer`, `Geod`, `always_xy=True`, UTM vs Web Mercator |
| **4. Measurements** | Project before measuring, `.area`, `.length`, geodesic distances, distance matrix |
| **5. Joins & Overlays** | `gpd.sjoin()`, `predicate=`, `gpd.overlay()`, self-join for proximity |
| **6. Fiona** | `fiona.open()`, schema inspection, per-feature reading, custom write |
| **7. Rasterio** | `rasterio.open()`, `.read()`, `from_bounds`, `sample_gen`, band stats |
| **8. Visualization** | `.plot()`, Folium `CircleMarker`, choropleth, save PNG + HTML |
| **🔬 Mini-Lab** | End-to-end pipeline: load → join → measure → report → map |

---

### ☑️ Self-assessment checklist

Before moving to Week 6, confirm you can do each of these **without looking at notes**:

- [ ] Create a `GeoDataFrame` from a list of coordinates
- [ ] Reproject a GDF to a local UTM zone
- [ ] Buffer city points by 50 km and measure the resulting areas
- [ ] Run a point-in-polygon spatial join
- [ ] Write a GeoDataFrame to GeoJSON and GeoPackage
- [ ] Open a raster with Rasterio and read band statistics
- [ ] Plot a GeoDataFrame with a colour column (choropleth)
- [ ] Build a Folium map with circle markers and popups

---

## 📚 Week 6 Preview — PostgreSQL Connection and Data Loading

Next week we move spatial data into a **production database**:

| Topic | What you will learn |
|-------|---------------------|
| **psycopg / SQLAlchemy** | Connect Python to PostgreSQL |
| **Schema design** | Tables for spatial data, data types |
| **Loading data** | Insert rows from GeoDataFrames and GeoJSON |
| **Transactions** | Safe bulk inserts with rollback on error |
| **PostGIS extension** | Enable and verify spatial types |

**Assignment due Week 7:**
Build an ETL loader that reads GeoJSON files, validates geometry, and inserts
clean records into a PostGIS table with logging and error recovery.

---

## 📖 Further Reading

| Resource | Why |
|----------|-----|
| [GeoPandas docs](https://geopandas.org/en/stable/) | API reference for all GDF operations |
| [Shapely manual](https://shapely.readthedocs.io/) | Geometry operations in depth |
| [PyProj docs](https://pyproj4.github.io/pyproj/) | CRS and transformer reference |
| [Fiona docs](https://fiona.readthedocs.io/) | Low-level vector I/O |
| [Rasterio docs](https://rasterio.readthedocs.io/) | Raster read/write/analysis |
| [Natural Earth Data](https://www.naturalearthdata.com/) | Free world admin/physical data |
| [Overpass Turbo](https://overpass-turbo.eu/) | Query OSM for local features |
| [EPSG.io](https://epsg.io/) | Look up any CRS by EPSG code |

---

*Geospatial Python Mastery — Week 5 of 10*
*For educational use. Please keep feedback to help improve future iterations.*